# Fraud detection on IEEE-CIS

Binary classification on the IEEE-CIS fraud dataset: ~590k card-not-present
e-commerce transactions, ~3.5% fraud, most features anonymised.

The focus here is the parts that matter for a model you'd actually deploy rather
than just score on a leaderboard: a time-based split, avoiding target leakage
while encoding categories, evaluation in cost terms (lift and a cost-based
threshold), and probability calibration + SHAP.

Data: IEEE-CIS Fraud Detection (Kaggle 2019, data from Vesta). On Kaggle, add it
through *Add Input* → `ieee-fraud-detection`.

## Load

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = '/kaggle/input/competitions/ieee-fraud-detection'
tx  = pd.read_csv(f'{DATA_DIR}/train_transaction.csv')
idf = pd.read_csv(f'{DATA_DIR}/train_identity.csv')

# identity only exists for some transactions -> left join
df = tx.merge(idf, how='left', on='TransactionID').sort_values('TransactionDT').reset_index(drop=True)
print(df.shape, '| fraud rate:', round(df['isFraud'].mean(), 4))

## 1. Class imbalance and time structure

In [ ]:
n = len(df)
n_fraud = df['isFraud'].sum()
print(f"transactions: {n:,}")
print(f"fraud: {n_fraud:,} ({n_fraud/n:.2%})")
print(f"accuracy of a 'never fraud' model: {1 - n_fraud/n:.2%}")

Only 3.5% of transactions are fraud, so accuracy is useless: a model that
predicts "no fraud" every time already scores 96.5%. Everything below uses
metrics that look at the rare class instead.

In [ ]:
# TransactionDT is a seconds offset -> day. Check whether the fraud rate drifts.
df['tx_day'] = (df['TransactionDT'] // 86400).astype(int)
daily = df.groupby('tx_day')['isFraud'].agg(['mean', 'count'])

fig, ax = plt.subplots(1, 2, figsize=(13, 3))
daily['mean'].plot(ax=ax[0], title='Fraud rate by day'); ax[0].set_ylabel('fraud share')
daily['count'].plot(ax=ax[1], title='Transactions by day')
plt.tight_layout(); plt.show()

Fraud rate isn't flat: around 2-3% for the first ~25 days, then a jump to
4-5% with spikes near 7%. Volume peaks around days 20 and 90 (shopping periods),
and the data covers roughly six months. So the data isn't stationary and the
patterns shift over time, which is the reason the split below is by time.

## 2. Time-based split

In [ ]:
cutoff = df['TransactionDT'].quantile(0.8)
train = df[df['TransactionDT'] <  cutoff].copy()
test  = df[df['TransactionDT'] >= cutoff].copy()

print(f"train: {len(train):>7,} rows (days {train['tx_day'].min()}-{train['tx_day'].max()})")
print(f"test:  {len(test):>7,} rows (days {test['tx_day'].min()}-{test['tx_day'].max()})")
print(f"fraud rate  train: {train['isFraud'].mean():.3%}  test: {test['isFraud'].mean():.3%}")

Train and test fraud rates are almost identical (3.51% vs 3.44%), but that
doesn't weaken the case for splitting by time. In production you always predict
forward, so training on future rows leaks information; and even with a stable
average, the feature combinations that flag fraud move across the six months. A
random split would give nicer but misleading numbers.

## 3. Feature engineering

Safe features first (computed per row, so they can't leak), then the encodings
that learn from the data and therefore have to be fit on train only.

In [ ]:
def basic_features(d):
    d = d.copy()
    d['amt_log'] = np.log1p(d['TransactionAmt'])
    d['amt_decimal'] = d['TransactionAmt'] - np.floor(d['TransactionAmt'])
    d['tx_hour'] = (d['TransactionDT'] // 3600) % 24
    d['n_missing'] = d.isna().sum(axis=1)
    return d

train = basic_features(train)
test  = basic_features(test)

# do round amounts have a higher fraud rate?
print(train.groupby(train['amt_decimal'] == 0)['isFraud'].mean())

Round amounts come out at 3.61% vs 3.41% — a weak signal. I keep the feature
but it isn't doing much on its own.

### Frequency encoding

Ground rule from here on: anything that learns from the data is fit on train and
only mapped onto test.

In [ ]:
cat_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'DeviceType']

freq_maps = {c: train[c].value_counts(normalize=True) for c in cat_cols}
for c in cat_cols:
    train[f'{c}_freq'] = train[c].map(freq_maps[c]).fillna(0.0)
    test[f'{c}_freq']  = test[c].map(freq_maps[c]).fillna(0.0)

print(train[['ProductCD', 'ProductCD_freq', 'card6', 'card6_freq']].head())

### Target encoding, and the leakage trap

Target encoding replaces a category with its mean fraud rate. It's strong, and
it's the most common way to leak. Both versions below, side by side.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

# leakage demo only - this column never goes into the model.
# mean over the whole dataset means each row sees its own label.
bad = df.groupby('card1')['isFraud'].mean()
df['card1_te_bad'] = df['card1'].map(bad)
print("AUC of this single leaked feature:", round(roc_auc_score(df['isFraud'], df['card1_te_bad']), 4))

One column scoring ~0.84 AUC is far too high — it has seen its own label.
Done properly (train only, with smoothing) it drops to a realistic test AUC.

In [ ]:
global_mean = train['isFraud'].mean()
smoothing = 20   # pull sparse categories toward the global mean

stats = train.groupby('card1')['isFraud'].agg(['mean', 'count'])
te_map = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)

train['card1_te'] = train['card1'].map(te_map).fillna(global_mean)
test['card1_te']  = test['card1'].map(te_map).fillna(global_mean)

print("train AUC:", round(roc_auc_score(train['isFraud'], train['card1_te']), 4))
print("test  AUC:", round(roc_auc_score(test['isFraud'],  test['card1_te']),  4))

Leaked 0.84 vs proper train 0.83 / test 0.75. The train-test gap is exactly
what the leaked version hides; 0.75 is the honest performance of one feature.

### Card pseudo-id

`card1 + addr1 + D1` approximates a single card. I skipped behavioural
aggregates (mean spend per card and so on) because the median history is about
two transactions per card, which is too little to build a stable pattern.

In [ ]:
for d in (train, test):
    d['card_uid'] = (d['card1'].astype('string').fillna('NA') + '_' +
                     d['addr1'].astype('string').fillna('NA') + '_' +
                     d['D1'].astype('string').fillna('NA'))

uid_freq = train['card_uid'].value_counts()
train['card_uid_freq'] = train['card_uid'].map(uid_freq).fillna(0)
test['card_uid_freq']  = test['card_uid'].map(uid_freq).fillna(0)

print("unique cards (train):", train['card_uid'].nunique())
print(train.groupby(train['card_uid_freq'] == 1)['isFraud'].mean())

## 4. Models

Baseline first, then LightGBM, so the gain from the stronger model is visible.
Both scored on the time-based test set.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

y_tr, y_te = train['isFraud'].values, test['isFraud'].values

# baseline: logistic regression on the engineered features
eng  = [c for c in train.columns if c.endswith('_freq') or c.endswith('_te')]
eng += ['amt_log', 'amt_decimal', 'tx_hour', 'n_missing']
eng  = [c for c in eng if c in train.columns]

lr = make_pipeline(SimpleImputer(strategy='median'),
                   StandardScaler(),
                   LogisticRegression(max_iter=1000, class_weight='balanced'))
lr.fit(train[eng], y_tr)
p_lr = lr.predict_proba(test[eng])[:, 1]
print(f"baseline logreg ({len(eng)} features)")
print("  PR-AUC :", round(average_precision_score(y_te, p_lr), 4))
print("  ROC-AUC:", round(roc_auc_score(y_te, p_lr), 4))

# main model: LightGBM on every numeric column
drop  = {'isFraud', 'TransactionID', 'TransactionDT', 'tx_day'}
feats = [c for c in train.select_dtypes(include=[np.number]).columns if c not in drop]

pos_w = (y_tr == 0).sum() / (y_tr == 1).sum()
model = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.03, num_leaves=64,
                           subsample=0.8, colsample_bytree=0.6,
                           scale_pos_weight=pos_w, random_state=42, n_jobs=-1)
model.fit(train[feats], y_tr)
p_lgb = model.predict_proba(test[feats])[:, 1]
print(f"\nlightgbm ({len(feats)} features)")
print("  PR-AUC :", round(average_precision_score(y_te, p_lgb), 4))
print("  ROC-AUC:", round(roc_auc_score(y_te, p_lgb), 4))

PR-AUC goes from ~0.14 (baseline) to ~0.53 (LightGBM), against a 0.035 base
rate. ROC-AUC barely moves (0.78 → 0.89) while PR-AUC nearly quadruples — with
96.5% negatives ROC-AUC flatters the model, so PR-AUC is the metric I rely on.
The baseline only sees the 11 engineered features and LightGBM sees all ~410, so
part of the gap is the extra data and part is the model.

## 5. Evaluation in cost terms

PR-AUC compares models but says nothing to a business. Lift and a cost-based
threshold turn the scores into operational numbers.

In [ ]:
# how much fraud we catch if we review the top X% by score
def gains_table(y_true, y_prob, amt, n_bins=10):
    order = np.argsort(-y_prob)
    y = np.asarray(y_true)[order]
    val = np.asarray(amt)[order]
    n, total_pos, total_val = len(y), y.sum(), val[y.astype(bool)].sum()
    rows = []
    for b in range(1, n_bins + 1):
        cut = int(np.ceil(n * b / n_bins))
        caught = y[:cut].sum()
        caught_val = val[:cut][y[:cut].astype(bool)].sum()
        rows.append({
            'top %': f"{b*10}%",
            'fraud caught': f"{caught/total_pos:.1%}",
            '$ caught': f"{caught_val/total_val:.1%}",
            'lift': round((caught/cut) / (total_pos/n), 2),
        })
    return pd.DataFrame(rows)

print(gains_table(y_te, p_lgb, test['TransactionAmt']).to_string(index=False))

In [ ]:
# choose the threshold by cost instead of the default 0.5
COST_FN = 150.0   # missed fraud: average loss on a fraudulent transaction
COST_FP = 5.0     # false alarm: manual review + customer friction

rows = []
for t in np.linspace(0.01, 0.99, 99):
    pred = (p_lgb >= t).astype(int)
    fn = int(((pred == 0) & (y_te == 1)).sum())
    fp = int(((pred == 1) & (y_te == 0)).sum())
    tp = int(((pred == 1) & (y_te == 1)).sum())
    rows.append({
        'threshold': round(t, 2),
        'recall': tp / (tp + fn),
        'precision': tp / (tp + fp) if (tp + fp) else 0,
        'flagged': pred.mean(),
        'cost': fn * COST_FN + fp * COST_FP,
    })
cc = pd.DataFrame(rows)

best = cc.loc[cc['cost'].idxmin()]
print(f"best threshold: {best['threshold']}")
print(f"  recall={best['recall']:.1%}  precision={best['precision']:.1%}  "
      f"flagged={best['flagged']:.1%}  cost=${best['cost']:,.0f}")

naive = cc.loc[cc['threshold'] == 0.50].iloc[0]
print(f"default 0.50:   recall={naive['recall']:.1%}  cost=${naive['cost']:,.0f}")

plt.figure(figsize=(7, 3))
plt.plot(cc['threshold'], cc['cost'])
plt.axvline(best['threshold'], color='r', ls='--', label=f"min at {best['threshold']}")
plt.xlabel('threshold'); plt.ylabel('expected cost ($)')
plt.title('Cost vs threshold'); plt.legend(); plt.show()

The cost curve is a U: too low a threshold floods review with false
positives, too high lets fraud through. The minimum sits around 0.32 and beats
the default 0.5 on both recall and total cost. Where it lands depends on the cost
matrix above, which is a business input, not a fixed constant.

## 6. Calibration and explainability

In [ ]:
from sklearn.calibration import calibration_curve

frac_pos, mean_pred = calibration_curve(y_te, p_lgb, n_bins=10, strategy='quantile')

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], 'k--', label='perfect')
plt.plot(mean_pred, frac_pos, 'o-', label='lightgbm')
plt.xlabel('predicted probability'); plt.ylabel('observed fraud rate')
plt.title('Reliability diagram'); plt.legend(); plt.show()

print(f"mean predicted: {p_lgb.mean():.3f}  |  actual fraud rate: {y_te.mean():.3f}")

`scale_pos_weight` helps the ranking but pushes probabilities up: mean
predicted 0.15 against an actual rate of 0.034. That matters because the cost
threshold above treats the scores as real probabilities.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator   # sklearn >= 1.6

# model is already trained; the calibrator must not see the test set
cal = CalibratedClassifierCV(FrozenEstimator(model), method='isotonic')
cal.fit(train[feats], y_tr)
p_cal = cal.predict_proba(test[feats])[:, 1]

fp_raw, mp_raw = calibration_curve(y_te, p_lgb, n_bins=10, strategy='quantile')
fp_cal, mp_cal = calibration_curve(y_te, p_cal, n_bins=10, strategy='quantile')

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], 'k--', label='perfect')
plt.plot(mp_raw, fp_raw, 'o-', label='raw')
plt.plot(mp_cal, fp_cal, 's-', label='calibrated')
plt.xlabel('predicted probability'); plt.ylabel('observed fraud rate')
plt.title('Calibration before / after'); plt.legend(); plt.show()

print(f"mean predicted after calibration: {p_cal.mean():.3f}  (target ~{y_te.mean():.3f})")
print(f"PR-AUC raw: {average_precision_score(y_te, p_lgb):.4f}  calibrated: {average_precision_score(y_te, p_cal):.4f}")

Isotonic calibration brings the mean prediction back to ~0.03 and PR-AUC is
basically unchanged (0.53 → 0.52) — the ranking is preserved while the
probabilities become usable for expected-loss decisions.

In [ ]:
import shap

# explain on a sample; the full test set would be slow
Xs = test[feats].sample(n=3000, random_state=42)

explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(Xs)

shap.summary_plot(sv, Xs, max_display=15, show=True)

The strongest feature is `card1_te`, the target-encoded card risk, and in
the expected direction: riskier cards push the score up. A couple of the other
hand-built features (email frequency, card-uid frequency) also land in the top
15 out of ~410. Many of the remaining drivers are anonymised V/C columns, so I
can show which features matter and in which direction but not their business
meaning — on real in-house data the same output would be directly
interpretable.

## Notes

- The cost-based threshold should be recomputed on the calibrated probabilities
  before any real use; 0.32 was fit on the raw scores.
- Possible extensions: out-of-fold target encoding inside a time-series CV, and
  per-card behavioural features if longer card histories were available.